# ViNumQA few-shot -- GLM-5.2 via FPT AI Factory

Separate copy of `vsf-few-shot-vinumqa.ipynb`, dedicated to GLM-5.2 on the FPT
AI Factory endpoint (`https://mkp-api.fptcloud.com`), which enforces
**RPM = 50** and **TPM = 100,000**. Kept as its own file rather than editing
the generic few-shot notebook so the rate-limiting/GLM-specific plumbing here
doesn't affect the other models run through that notebook.

Two things specific to GLM-5.2 on this endpoint, both verified empirically in
`vsf-few-shot-vinumqa-glm5.2-rate-check.ipynb`:
1. **Rate limits**: measured over 15 real probe requests, mean prompt_tokens=2407,
   mean total_tokens=2721/request -- TPM is the binding constraint (not RPM),
   capping this loop at ~36.7 req/min. Unthrottled, the loop hits 429s well
   before RPM=50 is reached, since ~37 requests already exhaust the 100k TPM
   budget. This notebook throttles via a sliding-window `RateLimiter`.
2. **Reasoning model**: GLM-5.2 returns its chain-of-thought in a separate
   `message.reasoning_content` field, distinct from `message.content` (the
   final program). Verified via a real probe: `content` came back as a clean
   `'add(2408, 1364)'` with no reasoning text mixed in, and `usage.completion_tokens`
   already includes the reasoning tokens. So `output = message.content` is
   correct as-is -- no extra parsing needed to strip reasoning out.

In [1]:
import pandas as pd
from pathlib import Path
from tabulate import tabulate
from openai import OpenAI

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".git").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

df = pd.read_json(PROJECT_ROOT / "datasets" / "ViNumQA" / "test.json")
df.sample(n=5)


,pre_text,table,post_text,id,qa
333,[chúng tôi có quyền chọn mua lợi ích loại a vớ...,"[[đơn vị: triệu đô la, tổng các khoản thanh to...",[tổng nghĩa vụ hợp đồng 12067.3 3112.0 3437.5 ...,GIS/2017/page_31.pdf-1,"{'question': 'Trong năm 2017, tỷ lệ phần trăm ..."
312,[chúng tôi đánh giá công ty cổ phần đầu tư và ...,"[[Tên dự án, Giá trị hiện tại thuần (triệu đồn...",[.],masvn/2020/2020044-CTCP_a__utu_va_Pha_ttrie__n...,{'question': 'Giá trị ròng của công ty (tổng g...
87,[mục 2 tài sản các văn phòng chính của chúng t...,"[[địa điểm, chức năng, diện tích ( feet vuông ...",[( 1 ) cơ sở tại woburn có tổng diện tích 163....,AMT/2007/page_29.pdf-4,{'question': 'phần nào của tài sản Woburn được...
216,"[định giá và khuyến nghị, chúng tôi nâng khuyế...","[[Năm tài chính (31/12), 2015, 2016, 2017, 201...",[.],masvn/2020/2020164-DIG_Q3Update_Tradingbuy_MAS...,{'question': 'Tổng doanh thu 3 dự án trong quý...
210,[chúng tôi sử dụng phương pháp so sánh để định...,"[[FY (Dec), FY 2015, FY 2016, FY 2017, FY 2018...",[.],masvn/2020/2020029-POW_FlashNote1H2019/page_1_QA5,"{'question': 'Dựa trên doanh thu và lãi gộp, b..."


In [2]:
len(df)


497

In [3]:
def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

def processing_input_question(sample):
    return sample["qa"]["question"]

def processing_program_content(sample):
    return sample["qa"]["program"]

def processing_answer_content(sample):
    return sample["qa"]["exe_ans"]

df["pre_text_processed"] = df.apply(lambda x: formatting_pre_text(x), axis=1)
df["post_text_processed"] = df.apply(lambda x: formatting_post_text(x), axis=1)
df["table_processed"] = df.apply(lambda x: formatting_table(x), axis=1)
df["table_raw"] = df["table"]  # keep raw rows for table_* row-name lookup at eval time
df["input_question"] = df.apply(lambda x: processing_input_question(x), axis=1)
df["program_processed"] = df.apply(lambda x: processing_program_content(x), axis=1)
df["answer_processed"] = df.apply(lambda x: processing_answer_content(x), axis=1)
df.sample(n=5)


,pre_text,table,post_text,id,qa,pre_text_processed,post_text_processed,table_processed,table_raw,input_question,program_processed,answer_processed
243,[được giao vào năm 2015 so với bảy chiếc được ...,"[[, 2016, 2015, 2014], [doanh thu thuần, $ 660...",[so sánh năm 2016 với năm 2015 doanh thu thuần...,LMT/2016/page_49.pdf-3,{'question': 'Doanh số bán hàng ròng trung bìn...,được giao vào năm 2015 so với bảy chiếc được g...,so sánh năm 2016 với năm 2015 doanh thu thuần ...,| | 2016 ...,"[[, 2016, 2015, 2014], [doanh thu thuần, $ 660...",Doanh số bán hàng ròng trung bình của MFC tron...,"table_average(doanh thu thuần, none)",6823.33333
84,[hoạt động chính trong lĩnh vực sản xuất giấy ...,"[[(Tỷ đồng), FY 2015, FY 2016, FY 2017, FY 201...",[.],masvn/2020/2020058-DHC_Companynote_MAS27.05.20...,{'question': 'EPS (thu nhập trên mỗi cổ phiếu)...,hoạt động chính trong lĩnh vực sản xuất giấy k...,.,| (Tỷ đồng) | FY 2015 | FY 2016 |...,"[[(Tỷ đồng), FY 2015, FY 2016, FY 2017, FY 201...",EPS (thu nhập trên mỗi cổ phiếu) của DHC tăng ...,"subtract(3383, 2512)",871.0
280,[hoạt động chính trong lĩnh vực sản xuất thuốc...,"[[(Tỷ đồng), FY 2016, FY 2017, FY 2018, FY 201...",[.],masvn/2020/2020192-LTG_Companynote_MAS15.12.20...,{'question': 'Tính tổng doanh thu từ năm 2016 ...,hoạt động chính trong lĩnh vực sản xuất thuốc ...,.,| (Tỷ đồng) | FY 2016 | FY 2017 |...,"[[(Tỷ đồng), FY 2016, FY 2017, FY 2018, FY 201...",Tính tổng doanh thu từ năm 2016 đến năm 2021(F...,"table_sum(Doanh thu, none)",47014.0
414,"[tập đoàn goldman sachs, inc. và các công ty c...","[[đơn vị: triệu đô la, cho ba tháng kết thúc h...","[trong bảng trên, các khoản mục ngoại bảng bao...",GS/2017/page_86.pdf-4,{'question': 'Sự thay đổi tính bằng triệu tron...,"tập đoàn goldman sachs, inc. và các công ty co...","trong bảng trên, các khoản mục ngoại bảng bao ...",| đơn vị: triệu đô la | cho ba ...,"[[đơn vị: triệu đô la, cho ba tháng kết thúc h...",Sự thay đổi tính bằng triệu trong các khoản ph...,"subtract(408164, 391555)",16609.0
119,[kế hoạch bồi thường được chủ sở hữu chứng kho...,"[[loại kế hoạch, số lượng chứng khoán sẽ được ...","[mục 13. các mối quan hệ nhất định, các giao d...",CME/2010/page_123.pdf-2,{'question': 'Giả sử tất cả các quyền chọn tro...,kế hoạch bồi thường được chủ sở hữu chứng khoá...,"mục 13. các mối quan hệ nhất định, các giao dị...",| loại kế hoạch ...,"[[loại kế hoạch, số lượng chứng khoán sẽ được ...",Giả sử tất cả các quyền chọn trong các kế hoạc...,"multiply(1211143, 308.10)",373153158.3


In [4]:
df = df[["pre_text_processed", "table_processed", "table_raw", "post_text_processed", "input_question", "program_processed", "answer_processed"]]
df.columns = [["pre_text", "table", "table_raw", "post_text", "question", "program", "answer"]]
df["generated_program"] = ""
df["calculated_program"] = ""
df


,pre_text,table,table_raw,post_text,question,program,answer,generated_program,calculated_program
0,thuyết minh báo cáo tài chính hợp nhất ( tiếp ...,| các thành phần của ảnh hưởng lũy kế của việc...,[[các thành phần của ảnh hưởng lũy kế của việc...,.,Sự thay đổi trong thu nhập ròng từ hiệu ứng tí...,"add(30, 1)",31.0,,
1,định giá và khuyến nghị:\nchúng tôi khuyến ngh...,| Năm tài chính (31/12) | FY17 | FY1...,"[[Năm tài chính (31/12), FY17, FY18, FY19, FY2...",.,"Theo dự phóng, doanh thu và lợi nhuận ròng quý...","subtract(9829, 642)",9187.0,,
2,"sử dụng phương pháp p/b và rnav để định giá, c...",| Năm tài chính (31/12) | 2016 | 2017 | ...,"[[Năm tài chính (31/12), 2016, 2017, 2018, 201...",.,IDC có bao nhiêu ha quỹ đất sẵn sàng cho thuê ...,"add(495, 398)",893.0,,
3,27/10/13 26/10/14 25/10/15 30/10/16 29/10/17 2...,| | 27/10/2013 |...,"[[, 27/10/2013, 26/10/2014, 25/10/2015, 30/10/...",.,Tỷ suất lợi nhuận trên đầu tư (ROI) của Applie...,"subtract(96.67, 100), divide(#0, 100)",-0.0333,,
4,thông tin tài chính bổ sung hiệu suất cổ phiếu...,| | 12/26/08 | ...,"[[, 12/26/08, 12/31/09, 12/31/10, 12/31/11, 12...",218 báo cáo thường niên năm 2013 của goldman s...,tỷ lệ lợi nhuận tích lũy tổng cộng theo phần t...,"subtract(248.36, 100), divide(#0, 100)",1.4836,,
...,...,...,...,...,...,...,...,...,...
492,thuyết minh báo cáo tài chính hợp nhất năm 201...,| ...,"[[, 2008, 2007], [Số dư đầu kỳ, $ 134.8, $ 266...",trong tổng số lợi ích thuế chưa được ghi nhận ...,Tỷ lệ phần trăm lợi ích thuế chưa được công nh...,"divide(131.8, 148.8)",0.88575,,
493,định giá và khuyến nghị:\nchúng tôi khuyến ngh...,| Năm tài chính (31/12) | FY17 | FY1...,"[[Năm tài chính (31/12), FY17, FY18, FY19, FY2...",.,Doanh thu trung bình từ năm tài chính 2017 đến...,"add(29710, 32662), add(#0, 35374), divide(#1, 3)",32582.0,,
494,định giá và khuyến nghị:\nchúng tôi khuyến ngh...,| Năm tài chính (31/12) | FY17 | FY1...,"[[Năm tài chính (31/12), FY17, FY18, FY19, FY2...",.,Tính phần trăm thay đổi EPS từ năm 2020 đến 2021.,"subtract(962, 788), divide(#0, 788)",0.22081,,
495,"trong quá trình kinh doanh thông thường, dựa t...",| ( đơn vị: nghìn ) | diện tích ròng chưa ph...,"[[( đơn vị: nghìn ), diện tích ròng chưa phát ...",( a ) một giếng khoan thăm dò được lên kế hoạc...,Tỷ lệ phần trăm diện tích đất chưa phát triển ...,"divide(145, 586)",0.24744,,


## Client + rate limiter setup

FPT AI Factory rate limits for GLM-5.2: RPM=50, TPM=100,000. Measured via
`vsf-few-shot-vinumqa-glm5.2-rate-check.ipynb` (15 real probe requests):
mean prompt_tokens=2407, mean total_tokens=2721/request -- TPM is the binding
constraint, capping this loop at ~36.7 req/min (one request every ~1.63s).

In [5]:
import os
import time
from collections import deque
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.environ["API_KEY"]
BASE_URL = os.environ["BASE_URL"]

# max_retries=0: disable the OpenAI SDK's own built-in retry-on-429 (it
# retries with its own back-off, invisible to RateLimiter below -- those
# retries hit the server without ever calling limiter.record(), so the
# limiter's window under-counts real requests/tokens sent, and the RPM cap
# gets blown from underneath it). All retry logic below is handled explicitly
# instead, so every request -- first try or retry -- goes through the limiter.
client = OpenAI(api_key=API_KEY, base_url=BASE_URL, max_retries=0)

MODEL = "GLM-5.2"

RPM_LIMIT = 50
TPM_LIMIT = 100_000


class RateLimiter:
    """Sliding 60s window limiter for RPM and TPM.

    Call `wait_if_needed(estimated_tokens)` before each request (blocks via
    time.sleep if the window is already at capacity), then `record(actual_tokens)`
    after the response arrives with the real usage.total_tokens.
    """

    def __init__(self, rpm_limit: int, tpm_limit: int, window_s: float = 60.0):
        self.rpm_limit = rpm_limit
        self.tpm_limit = tpm_limit
        self.window_s = window_s
        self.request_times = deque()
        self.token_events = deque()

    def _prune(self, now: float):
        while self.request_times and now - self.request_times[0] > self.window_s:
            self.request_times.popleft()
        while self.token_events and now - self.token_events[0][0] > self.window_s:
            self.token_events.popleft()

    def wait_if_needed(self, estimated_tokens: int):
        while True:
            now = time.monotonic()
            self._prune(now)

            tokens_in_window = sum(t for _, t in self.token_events)
            requests_in_window = len(self.request_times)

            over_rpm = requests_in_window >= self.rpm_limit
            over_tpm = tokens_in_window + estimated_tokens > self.tpm_limit

            if not over_rpm and not over_tpm:
                return

            oldest = min(
                self.request_times[0] if self.request_times else float("inf"),
                self.token_events[0][0] if self.token_events else float("inf"),
            )
            sleep_for = max(0.05, self.window_s - (now - oldest))
            time.sleep(sleep_for)

    def record(self, actual_tokens: int):
        now = time.monotonic()
        self.request_times.append(now)
        self.token_events.append((now, actual_tokens))

    def on_rate_limit_error(self, retry_after=None):
        """Force the window to look full for retry_after seconds (or the
        whole window if the server didn't give a Retry-After hint), so the
        next wait_if_needed() blocks appropriately -- the 429 itself is proof
        the local window's accounting has drifted from the server's real
        state (e.g. because of a prior SDK-level retry, or another process
        sharing the same API key), so the safest correction is to treat the
        window as saturated rather than trust the (evidently wrong) count.
        """
        now = time.monotonic()
        self.request_times.clear()
        self.token_events.clear()
        pause = retry_after if retry_after is not None else self.window_s
        self.request_times.append(now + pause - self.window_s)
        self.token_events.append((now + pause - self.window_s, self.tpm_limit))


limiter = RateLimiter(rpm_limit=RPM_LIMIT, tpm_limit=TPM_LIMIT)

In [6]:
SYSTEM_MESSAGE = """You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.

### LIST OF 10 VALID OPERATORS:

1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(row_name, none) -> sum of the numeric values in the table row named `row_name`
8. table_average(row_name, none) -> arithmetic mean of the numeric values in the table row named `row_name`
9. table_max(row_name, none) -> maximum of the numeric values in the table row named `row_name`
10. table_min(row_name, none) -> minimum of the numeric values in the table row named `row_name`

### RULES:
- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.
- table_* operators take exactly two arguments: the row name (copied exactly as it appears as the first cell of the target row) and the literal `none` (e.g. table_max(Lãi ròng, none)), never a list of numeric values.
- Do not perform mental calculations or provide explanations. The output must contain only the program string.
- Reference the result of a previous step using #0 (step 1), #1 (step 2), etc. Steps are separated by commas.
- Preserve the original number format from the context. If a value is missing, use 'none'."""

USER_MESSAGE_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}

[TABLE]
{table}

[TEXT AFTER TABLE]
{post_text}

### QUESTION:
{question}

### PROGRAM:"""

# 3 few-shot demonstrations sampled from train.json, one per evidence type
# (Table Only / Text Only / Table & Text), following the ViNumQA task's
# own categorization (see VLSP 2025 NumQA paper, Section 3.3).
# The "table" field of each shot is pre-rendered to the same GitHub-markdown
# format that `formatting_table` produces for the real data, so the few-shot
# demonstrations are formatted identically to the actual queries.
FEW_SHOT_EXAMPLES = [
    {
        # Table Only (train idx 1959): answer is derived purely from two table cells.
        "pre_text": "phụ lục iv ace limited và các công ty con thông tin bổ sung về phí tái bảo hiểm thu được cho các năm kết thúc ngày 31 tháng 12 năm 2010, 2009 và 2008 (tính bằng triệu đô la mỹ, ngoại trừ tỷ lệ phần trăm) số tiền trực tiếp nhượng cho các công ty nhận từ các công ty khác số tiền ròng tỷ lệ phần trăm số tiền nhận được trên.",
        "table": (
            "|   cho các năm kết thúc ngày 31 tháng 12 năm 2010, 2009 và 2008 (tính bằng triệu đô la Mỹ, ngoại trừ tỷ lệ phần trăm) | số tiền trực tiếp   | nhượng cho các công ty khác   | nhận từ các công ty khác   | số tiền ròng   | tỷ lệ phần trăm số tiền nhận được trên số tiền ròng   |\n"
            "|----------------------------------------------------------------------------------------------------------------------|---------------------|-------------------------------|----------------------------|----------------|-------------------------------------------------------|\n"
            "|                                                                                                                 2010 | $ 15780             | $ 5792                        | $ 3516                     | $ 13504        | 26% ( 26 % )                                          |\n"
            "|                                                                                                                 2009 | $ 15415             | $ 5943                        | $ 3768                     | $ 13240        | 28% ( 28 % )                                          |\n"
            "|                                                                                                                 2008 | $ 16087             | $ 6144                        | $ 3260                     | $ 13203        | 25% ( 25 % )                                          |"
        ),
        "post_text": ".",
        "question": "Sự khác biệt giữa số tiền chuyển giao và nhận chuyển giao trong năm 2010 là bao nhiêu?",
        "program": "subtract(5792, 3516)",
    },
    {
        # Text Only (train idx 93): the supplied table (VHM financial summary) is
        # irrelevant to the question; the program only uses numbers from pre_text.
        "pre_text": "hệ số khả năng thanh toán lãi vay cũng tăng cao đạt mức 13.2 lần, so với chỉ 10.3 lần cùng kỳ.",
        "table": (
            "|                   |   FY 2015 |   FY 2016 |   FY 2017 |   FY 2018 |   FY 2019(F) |\n"
            "|-------------------|-----------|-----------|-----------|-----------|--------------|\n"
            "| Doanh thu (VNDbn) |      4920 |     11217 |     15297 |     38664 |        71115 |\n"
            "| Lãi gộp (Vbn)     |       718 |      2420 |      3128 |      7617 |        10983 |"
        ),
        "post_text": ".",
        "question": "Hệ số khả năng thanh toán lãi vay tăng bao nhiêu lần so với cùng kỳ năm ngoái?",
        "program": "subtract(13.2, 10.3)",
    },
    {
        # Table & Text (train idx 1127): must locate the right table row ("Nội dung số")
        # across three columns and chain two operators via the #0 reference.
        "pre_text": "tỷ lệ phần trăm chi phí vốn trên phần trăm tổng tài sản của mảng viễn thông được duy trì trên 1, cho thấy sự tập trung phân bổ chi phí vốn vào mảng viễn thông của fpt qua các năm.\nngoài ra, tỷ lệ này của mảng đầu tư và giáo dục là 1,1 vào năm 2018 và 0,9 vào năm 2019, khẳng định fpt cũng đang tập trung vào phát triển 2 mảng này trong 2 năm gần đây.",
        "table": (
            "|                     |   2014 |   2015 |   2016 |   2017 |   2018 |   2019 |\n"
            "|---------------------|--------|--------|--------|--------|--------|--------|\n"
            "| Viễn thông          |    1.8 |    2.4 |    1.9 |    1.4 |    1.7 |    1.8 |\n"
            "| Nội dung số         |    0.4 |    0.2 |    0.9 |    0.1 |    0.1 |    0.1 |\n"
            "| Phát triển phần mềm |    2.4 |    1.3 |    3.1 |    1.1 |    0.4 |    0.5 |"
        ),
        "post_text": ".",
        "question": "Tổng tỷ lệ của mảng Nội dung số trong ba năm từ 2014 đến 2016 là bao nhiêu?",
        "program": "add(0.4, 0.2), add(#0, 0.9)",
    },
]


## Generate programs (rate-limited, resumable)

Same generation shape as the base few-shot notebook, with additions for this
endpoint:
- `RateLimiter` throttles requests before they're sent (a rough chars/4
  pre-estimate is enough since the window is corrected with the real
  `usage.total_tokens` right after).
- `output` reads `message.content` only -- GLM-5.2's reasoning trace lives in
  `message.reasoning_content`, kept separate by the API itself, so no extra
  stripping is needed here.
- **Checkpointing**: results are saved to a JSON file after every request, and
  reloaded at the start -- already-generated samples are skipped, so a 429 (or
  any other interruption) partway through doesn't require regenerating from
  sample 0. This is what actually happened on the first real run: it died at
  73/497 on a `RateLimitError` (see the retry logic below for why that
  shouldn't recur, but the checkpoint means even if it does, nothing already
  generated is lost).
- **Explicit retry-with-backoff on `RateLimitError`**: the OpenAI SDK's own
  retry (now disabled via `max_retries=0` above) issued retries the
  `RateLimiter` never saw, which is the most likely reason the limiter's
  internal RPM count silently drifted below what was actually sent to the
  server -- the observed 429 said "RPM exceeded" even though the limiter's
  own accounting showed far below 50 req/min at the time. On a 429 here, the
  server's `Retry-After` header (or a 65s default) is used to both sleep and
  reset the limiter's window via `on_rate_limit_error`, then the same sample
  is retried -- up to 5 attempts before giving up on that sample and moving on.

In [13]:
import json
import re
from tqdm import tqdm
from openai import RateLimitError

CHECKPOINT_PATH = PROJECT_ROOT / "notebooks" / "vinumqa" / "few-shot" / "vsf-few-shot-vinumqa-glm5.2.checkpoint.json"
MAX_RETRIES_PER_SAMPLE = 5
DEFAULT_RETRY_AFTER_S = 65.0  # the 429 we hit said "try again in 60s" -- pad slightly

# Build the fixed few-shot prefix once: alternating user/assistant turns,
# one pair per FEW_SHOT_EXAMPLES entry, each following the same USER_MESSAGE_FRAME
# used for the real query so the model sees a consistent input/output format.
few_shot_messages = []
for shot in FEW_SHOT_EXAMPLES:
    few_shot_messages.append({
        "role": "user",
        "content": USER_MESSAGE_FRAME.format(
            pre_text=shot["pre_text"],
            table=shot["table"],
            post_text=shot["post_text"],
            question=shot["question"],
        )
    })
    few_shot_messages.append({"role": "assistant", "content": shot["program"]})

MAX_TOKENS = 8192

# Resume support: load any previously-generated outputs (keyed by the
# dataframe's integer index, saved as a string in JSON) and skip those rows
# below instead of regenerating them.
if CHECKPOINT_PATH.exists():
    with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
        checkpoint = {int(k): v for k, v in json.load(f).items()}
    print(f"Resuming from checkpoint: {len(checkpoint)} samples already generated.")
else:
    checkpoint = {}

for df_index, output in checkpoint.items():
    df.at[df_index, "generated_program"] = output

def _extract_retry_after(err):
    header_val = err.response.headers.get("retry-after") if err.response is not None else None
    if header_val is not None:
        try:
            return float(header_val)
        except ValueError:
            pass
    # Fallback: the FPT AI Factory error body embeds it in the message text,
    # e.g. "...Please try again in 60s." -- same shape as the 429 we hit.
    match = re.search(r"try again in (\d+(?:\.\d+)?)s", str(err))
    return float(match.group(1)) if match else None

pending_indices = [idx for idx in df.index if idx not in checkpoint]

for df_index in tqdm(pending_indices, desc="Generating program..."):
    values = df.loc[df_index]
    pre_text = values["pre_text"]
    table = values["table"]
    post_text = values["post_text"]
    question = values["question"]

    messages = [
        {
            "role": "system",
            "content": SYSTEM_MESSAGE,
        },
        *few_shot_messages,
        {
            "role": "user",
            "content": USER_MESSAGE_FRAME.format(pre_text=pre_text, table=table, post_text=post_text, question=question)
        },
    ]

    # Estimate tokens before the call (real usage.prompt_tokens isn't known
    # until the response arrives) -- rough chars/4 heuristic is enough here
    # since limiter.record() below corrects the window with the real total
    # right after, so any under/over-estimate only affects this one wait.
    est_tokens = sum(len(m["content"]) for m in messages) // 4 + MAX_TOKENS

    output = None
    for attempt in range(1, MAX_RETRIES_PER_SAMPLE + 1):
        limiter.wait_if_needed(est_tokens)
        try:
            chat_completion = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                temperature=0.0,
                max_tokens=MAX_TOKENS,
                stream=False,
            )
        except RateLimitError as err:
            retry_after = _extract_retry_after(err) or DEFAULT_RETRY_AFTER_S
            print(f"\n[df_index={df_index}] RateLimitError on attempt {attempt}/{MAX_RETRIES_PER_SAMPLE}: "
                  f"{err}. Sleeping {retry_after:.0f}s and resetting the limiter window before retrying.")
            limiter.on_rate_limit_error(retry_after)
            time.sleep(retry_after)
            continue

        limiter.record(chat_completion.usage.total_tokens)

        # message.content is the final program string; GLM-5.2's chain-of-thought
        # is returned separately in message.reasoning_content, not mixed in here.
        output = chat_completion.choices[0].message.content.strip().strip("\n")
        break

    if output is None:
        print(f"\n[df_index={df_index}] Giving up after {MAX_RETRIES_PER_SAMPLE} attempts -- leaving generated_program empty.")
        output = ""

    df.at[df_index, "generated_program"] = output
    checkpoint[df_index] = output
    with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in checkpoint.items()}, f, ensure_ascii=False, indent=2)

    gold_program = values["program"]
    # print(f"TEST SAMPLE {df_index}:\n\nPREDICTION:\n{output}\n\nGROUND_TRUTH:\n{gold_program}")
    # print("="*100)

Resuming from checkpoint: 491 samples already generated.


Generating program...:   0%|          | 0/6 [00:00<?, ?it/s]


[df_index=491] RateLimitError on attempt 1/5: Error code: 429 - {'subcode': 429000, 'code': 429, 'description': 'Rate limit exceeded for GLM-5.2 on RPM. Your maximum thresholds are 50 RPM. Please try again in 60s.', 'message': 'Too Many Requests'}. Sleeping 60s and resetting the limiter window before retrying.


Generating program...:  33%|███▎      | 2/6 [01:12<02:02, 30.71s/it]


[df_index=493] RateLimitError on attempt 1/5: Error code: 429 - {'message': 'Too Many Requests', 'description': 'Rate limit exceeded for GLM-5.2 on RPM. Your maximum thresholds are 50 RPM. Please try again in 60s.', 'code': 429, 'subcode': 429000}. Sleeping 60s and resetting the limiter window before retrying.


Generating program...:  50%|█████     | 3/6 [02:22<02:25, 48.53s/it]


[df_index=494] RateLimitError on attempt 1/5: Error code: 429 - {'subcode': 429000, 'description': 'Rate limit exceeded for GLM-5.2 on RPM. Your maximum thresholds are 50 RPM. Please try again in 60s.', 'code': 429, 'message': 'Too Many Requests'}. Sleeping 60s and resetting the limiter window before retrying.

[df_index=494] RateLimitError on attempt 2/5: Error code: 429 - {'message': 'Too Many Requests', 'description': 'Rate limit exceeded for GLM-5.2 on RPM. Your maximum thresholds are 50 RPM. Please try again in 60s.', 'code': 429, 'subcode': 429000}. Sleeping 60s and resetting the limiter window before retrying.


Generating program...: 100%|██████████| 6/6 [04:35<00:00, 45.87s/it]


In [14]:
df

,pre_text,table,table_raw,post_text,question,program,answer,generated_program,calculated_program
0,thuyết minh báo cáo tài chính hợp nhất ( tiếp ...,| các thành phần của ảnh hưởng lũy kế của việc...,[[các thành phần của ảnh hưởng lũy kế của việc...,.,Sự thay đổi trong thu nhập ròng từ hiệu ứng tí...,"add(30, 1)",31.0,"add(-54, 30), add(#0, 1)",
1,định giá và khuyến nghị:\nchúng tôi khuyến ngh...,| Năm tài chính (31/12) | FY17 | FY1...,"[[Năm tài chính (31/12), FY17, FY18, FY19, FY2...",.,"Theo dự phóng, doanh thu và lợi nhuận ròng quý...","subtract(9829, 642)",9187.0,"subtract(9829, 642)",
2,"sử dụng phương pháp p/b và rnav để định giá, c...",| Năm tài chính (31/12) | 2016 | 2017 | ...,"[[Năm tài chính (31/12), 2016, 2017, 2018, 201...",.,IDC có bao nhiêu ha quỹ đất sẵn sàng cho thuê ...,"add(495, 398)",893.0,"add(495, 398)",
3,27/10/13 26/10/14 25/10/15 30/10/16 29/10/17 2...,| | 27/10/2013 |...,"[[, 27/10/2013, 26/10/2014, 25/10/2015, 30/10/...",.,Tỷ suất lợi nhuận trên đầu tư (ROI) của Applie...,"subtract(96.67, 100), divide(#0, 100)",-0.0333,"subtract(96.67, 100), divide(#0, 100)",
4,thông tin tài chính bổ sung hiệu suất cổ phiếu...,| | 12/26/08 | ...,"[[, 12/26/08, 12/31/09, 12/31/10, 12/31/11, 12...",218 báo cáo thường niên năm 2013 của goldman s...,tỷ lệ lợi nhuận tích lũy tổng cộng theo phần t...,"subtract(248.36, 100), divide(#0, 100)",1.4836,"subtract(248.36, 100.00), divide(#0, 100.00), ...",
...,...,...,...,...,...,...,...,...,...
492,thuyết minh báo cáo tài chính hợp nhất năm 201...,| ...,"[[, 2008, 2007], [Số dư đầu kỳ, $ 134.8, $ 266...",trong tổng số lợi ích thuế chưa được ghi nhận ...,Tỷ lệ phần trăm lợi ích thuế chưa được công nh...,"divide(131.8, 148.8)",0.88575,"divide(131.8, 148.8), multiply(#0, 100)",
493,định giá và khuyến nghị:\nchúng tôi khuyến ngh...,| Năm tài chính (31/12) | FY17 | FY1...,"[[Năm tài chính (31/12), FY17, FY18, FY19, FY2...",.,Doanh thu trung bình từ năm tài chính 2017 đến...,"add(29710, 32662), add(#0, 35374), divide(#1, 3)",32582.0,"add(29710, 32662), add(#0, 35374), divide(#1, 3)",
494,định giá và khuyến nghị:\nchúng tôi khuyến ngh...,| Năm tài chính (31/12) | FY17 | FY1...,"[[Năm tài chính (31/12), FY17, FY18, FY19, FY2...",.,Tính phần trăm thay đổi EPS từ năm 2020 đến 2021.,"subtract(962, 788), divide(#0, 788)",0.22081,"subtract(962, 788), divide(#0, 788), multiply(...",
495,"trong quá trình kinh doanh thông thường, dựa t...",| ( đơn vị: nghìn ) | diện tích ròng chưa ph...,"[[( đơn vị: nghìn ), diện tích ròng chưa phát ...",( a ) một giếng khoan thăm dò được lên kế hoạc...,Tỷ lệ phần trăm diện tích đất chưa phát triển ...,"divide(145, 586)",0.24744,"divide(145, 586)",


In [15]:
import sys
sys.path.insert(0, "../../evaluate")  # fallback: relative path when running locally

from scorer import evaluate_dataframe  # noqa: E402

# scorer.py is the shared ViNumQA evaluator (notebooks/evaluate/scorer.py): it
# ports FinQA's official evaluation protocol (sympy-based symbolic Program
# Accuracy, table-row-lookup-aware Execution Accuracy) instead of a
# hand-rolled parser, and correctly executes table_*(row_name, none) calls by
# looking up the named row in the raw table -- which the previous in-notebook
# parser could not do at all (it treated table_* arguments as raw numbers).


In [17]:
df_scored, summary = evaluate_dataframe(
    df,
    generated_col="generated_program",   # cột bạn đang ghi output model vào
    gold_program_col="program",
    gold_answer_col="answer",
    table_col="table_raw",
)

print(summary)  # {'program_accuracy': ..., 'execution_accuracy': ...}


{'program_accuracy': 0.5090543259557344, 'execution_accuracy': 0.5553319919517102}
